<a href="https://colab.research.google.com/github/vtminc1000/Analisis-Sentimen/blob/main/TextBlob.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pandas openpyxl textblob googletrans==4.0.0-rc1 matplotlib -q

In [ ]:
import pandas as pd
import re
from google.colab import files
from textblob import TextBlob
from googletrans import Translator
import matplotlib.pyplot as plt

In [ ]:
uploaded = files.upload()

In [ ]:
df = pd.read_excel('Data_Inite.xlsx')
df.head()

In [ ]:
data_long = []

for _, row in df.iterrows():
    data_long.append({'responden_id': row['responden_id'], 'tahap': 'pre', 'pertanyaan': 'q1', 'teks': row['pre_q1'], 'label_asli': row['pre_q1_label']})
    data_long.append({'responden_id': row['responden_id'], 'tahap': 'pre', 'pertanyaan': 'q2', 'teks': row['pre_q2'], 'label_asli': row['pre_q2_label']})
    data_long.append({'responden_id': row['responden_id'], 'tahap': 'post', 'pertanyaan': 'q1', 'teks': row['post_q1'], 'label_asli': row['post_q1_label']})
    data_long.append({'responden_id': row['responden_id'], 'tahap': 'post', 'pertanyaan': 'q2', 'teks': row['post_q2'], 'label_asli': row['post_q2_label']})

data_long = pd.DataFrame(data_long)

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\s+', ' ', text).strip()
    return text

data_long['teks_clean'] = data_long['teks'].apply(clean_text)

In [ ]:
translator = Translator()

In [ ]:
def translate_text(text):
    try:
        return translator.translate(text, src='id', dest='en').text
    except:
        return text

In [ ]:
data_long['teks_en'] = data_long['teks_clean'].apply(translate_text)

In [ ]:
def textblob_label(text):
    polarity = TextBlob(text).sentiment.polarity

    if polarity > 0:
        return 1
    elif polarity < 0:
        return -1
    else:
        return 0

In [ ]:
data_long['textblob_label'] = data_long['teks_en'].apply(textblob_label)

In [ ]:
data_long[['teks', 'teks_en', 'textblob_label']].head(10)

In [ ]:
data_q1 = data_long[data_long['pertanyaan'] == 'q1']

distribusi_q1 = data_q1.groupby(['tahap', 'textblob_label']).size().unstack(fill_value=0)

distribusi_q1 = distribusi_q1.rename(columns={
    -1: 'Negatif',
    0: 'Netral',
    1: 'Positif'
})

distribusi_q1

In [ ]:
data_q2 = data_long[data_long['pertanyaan'] == 'q2']

distribusi_q2 = data_q2.groupby(['tahap', 'textblob_label']).size().unstack(fill_value=0)

distribusi_q2 = distribusi_q2.rename(columns={
    -1: 'Negatif',
    0: 'Netral',
    1: 'Positif'
})

distribusi_q2

In [ ]:
plt.figure()
distribusi_q1.plot(kind='bar')
plt.title('Distribusi Sentimen Q1 (TextBlob)')
plt.xlabel('Tahap')
plt.ylabel('Jumlah')
plt.xticks(rotation=0)

plt.savefig('grafik_q1_textblob.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure()
distribusi_q2.plot(kind='bar')
plt.title('Distribusi Sentimen Q2 (TextBlob)')
plt.xlabel('Tahap')
plt.ylabel('Jumlah')
plt.xticks(rotation=0)

plt.savefig('grafik_q2_textblob.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
data_long.to_excel('hasil_textblob.xlsx', index=False)
distribusi_q1.to_excel('distribusi_q1_textblob.xlsx')
distribusi_q2.to_excel('distribusi_q2_textblob.xlsx')

In [ ]:
files.download('hasil_textblob.xlsx')
files.download('distribusi_q1_textblob.xlsx')
files.download('distribusi_q2_textblob.xlsx')
files.download('grafik_q1_textblob.png')
files.download('grafik_q2_textblob.png')

In [ ]:
data_long[['teks', 'teks_en', 'textblob_label']].head(10)